# Zonal Analysis

In [ ]:
import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio as rio
import swisslandstats as sls
from shapely.geometry import Point

import pylandstats as pls

Landscapes tend to be heterogeneous and complex and therefore reducing such information to a single scalar value for all the landscape often leads to metric values that are hard to interpret. It might thus be helpful to decompose the landscape into a set of zones of interest and compute the metrics for each zone separately. Such approach to GIS is often referred to as zonal analysis. The pylandstats library features three classes that might be used to that end: the more generic `ZonalAnalysis`, `BufferAnalysis` and `ZonalGridAnalysis`.

The data used in this notebook ships with the docs in the `data` directory, namely:
- the land use/land cover (LULC) data is downloaded and preprocessed (see [A03-swisslandstats-preprocessing.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/A03-swisslandstats-preprocessing.ipynb) for more details).
- the elevation zones vector data is downloaded and preprocessed (see [A04-elevation-zones.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/A04-elevation-zones.ipynb) for more details).

Consider the following landscape of the District of the Veveyse, Switzerland:

In [ ]:
URBAN_CLASS_VAL = 1
input_filepath = "data/veveyse/AS18_4.tif"

with rio.open(input_filepath) as src:
    plt.imshow(src.read(1), cmap=sls.noas04_4_cmap, norm=sls.noas04_4_norm)

## Zonal analysis

The `ZonalAnalysis` class of PyLandStats serves to compute the landscape metrics over any set of zones. Instantiating a `ZonalAnalysis` requires two positional arguments, namely the landscape file and the zones, which can be defined by means of vector geometries or a labelled array of the same form as the landscape (i.e., mapping each landscape pixel to its zone).

### Using vector geometries to define zones

The most straight-forward way to define a zonal analysis is to use a geopandas geo-series or geo-data frame or its equivalent file-like object. In this example, we will use a geopackage file defining a set of elevation zones. Additionally, we will use the `zone_index` argument to indicate which column of the geopackage file will be used to index the zones:

In [ ]:
elev_zones_filepath = "data/elev-zones.gpkg"

za = pls.ZonalAnalysis(input_filepath, elev_zones_filepath, zone_index="elev-zone")

The `ZonalAnalysis` instance will automatically generate the three landscapes of interest (one for each transect) by masking the pixels of the input raster. Such information is stored as part of the `zone_gser` attribute:

In [ ]:
za.zone_gser

We can also plot the `zone_gser` attribute to visualize the zones (we are resetting the index so that we can use it as color scheme) with a basemap (obtained using the [contextily](https://github.com/geopandas/contextily) library):

In [ ]:
ax = za.zone_gser.reset_index().plot(
    "elev-zone", alpha=0.6, categorical=True, legend=True
)
cx.add_basemap(ax, crs=za.zone_gser.crs, source=cx.providers.CartoDB.Positron)

Similarly, we can visualize the landscape rasters of each zone by means of the `plot_landscapes` method as in:

In [ ]:
fig = za.plot_landscapes(
    cmap=sls.noas04_4_cmap, show_kwargs=dict(norm=sls.noas04_4_norm)
)

The goal is now to compute and plot the landscape metrics for each zone. Like `SpatioTemporalAnalysis`, `ZonalAnalysis` only supports class and landscape-level metrics, which again, can be computed by means of its methods `compute_class_metrics_df` and `compute_landscape_metrics_df` respectively, e.g.:

In [ ]:
za.compute_class_metrics_df()

(customizing-zonal-analysis)=
Likewise `SpatioTemporalAnalysis`, if we want to compute the metrics data frame only for a subset of metrics or classes, or customize how the metrics are computed, we must pass the arguments `metrics`, `classes` or `metrics_kwargs` to the `compute_class_metrics_df` and `compute_landscape_metrics_df`, as in:

In [ ]:
metrics = ["proportion_of_landscape", "edge_density", "fractal_dimension_am"]
classes = [URBAN_CLASS_VAL]
metrics_kwargs = {
    "proportion_of_landscape": {"percent": False},
    "edge_density": {"count_boundary": True},
}
za.compute_class_metrics_df(
    metrics=metrics, classes=[URBAN_CLASS_VAL], metrics_kwargs=metrics_kwargs
)

On the other hand, the `plot_metric` method of `ZonalAnalysis` will plot the value of a given metric for each zone:

In [ ]:
za.plot_metric("proportion_of_landscape", class_val=URBAN_CLASS_VAL)

In this case we see how the proportion of urbanized landscape (class `1`) becomes zero at highest altitudes.

Finally, in order to visualize such information in space, the zonal statistics can be computed in the form of a geo-data frame with the `compute_zonal_statistics_gdf` method as in:

In [ ]:
zonal_statistics_gdf = za.compute_zonal_statistics_gdf(
    metrics=metrics, class_val=URBAN_CLASS_VAL
)
zonal_statistics_gdf.head()

In [ ]:
zonal_statistics_gdf.plot("edge_density")

The computed metrics are essentially the same as those obtained using the `compute_class_metrics_df` or `compute_landscape_metrics_df` (depending on whether a `class_val` argument is provided or not), with an additional column featuring the vector geometry of each zone. This actually corresponds to a geopandas geo-data frame, and as such, we can use [its `geopandas.GeoDataFrame.plot` method](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html) to spatially visualize the value of a metric across the zones:

In [ ]:
zonal_statistics_gdf.plot("proportion_of_landscape", legend=True)

### Using labelled raster arrays to define zones

The zones can be defined as NumPy labelled arrays of the same shape of the landscape of interest, where each zone is labelled by a unique integer value, which will be used to identify the zones (i.e., as index).

For instance, in order to compute the metrics over a set of rectangular transects, let us define three transects of 50x50 cells (5x5km) that horizontally traverse our landscape at the latitude of Chatel-St-Denis:

In [ ]:
with rio.open(input_filepath) as src:
    label_arr = np.zeros(src.shape, dtype=np.uint8)

transect_len = 50
for i, transect_start in enumerate(range(0, 150, transect_len), start=1):
    label_arr[80:130, transect_start : transect_start + transect_len] = i

plt.imshow(label_arr)
plt.colorbar()

As we can see, the array labels each transect zone by a unique integer. We can use it as the `zones` argument when initializing a `ZonalAnalysis` instance:

In [ ]:
za = pls.ZonalAnalysis(input_filepath, label_arr)
za.compute_zonal_statistics_gdf(
    metrics=["proportion_of_landscape"], class_val=URBAN_CLASS_VAL
)

Note from the geo-data frame above that the label values are used to index the zones. We can override this behaviour by providing a custom `zone_index` argument, e.g., strings that denote that each landscape corresponds to the transect from kilometers 0 to 5, 5 to 10 and 10 to 15 respectivel:

In [ ]:
za = pls.ZonalAnalysis(input_filepath, label_arr, zone_index=["0-5", "5-10", "10-15"])
za.compute_zonal_statistics_gdf(
    metrics=["proportion_of_landscape"], class_val=URBAN_CLASS_VAL
)

We can see that the highest proportion of urban land cover is found in the western and central transects, which actually correspond to the town of Chatel-St-Denis.

## Buffer Analysis

In line with the classic concentric models of location and land use, we are often interested in evaluate how the landscape patterns change as we move away from the urban center. This is similar to the "gradient analysis" approach from landscape ecology, which consists in evaluating the spatial variation of the landscape patterns as one moves progressively from the highly-developed urban cores to the less intense suburbs until the rural and natural hinterlands.

To that end, PyLandStats features the `BufferAnalysis` class (which inherits from the `ZonalAnalysis` class), which defines a series of spatial extents for our landscape based on buffers of increasing distances around our feature of interest - in this example, the town of Chatel-St-Denis.

### From Point

We might define a given coordinate as the center of our region of interest (in this example, the town of Chatel-St-Denis) and a series of buffer distances around that point:

In [ ]:
# latitude and longitude of the center of Chatel-St-Denis according to OpenStreetMap
base_geom = Point(6.8992073, 46.52634)
base_geom_crs = "+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs"

# buffer distances (in meters)
buffer_dists = [2000, 4000, 6000]

then, we can use the `BufferAnalysis` class of Pylandstats as in (note that in this case we need to provide the CRS of the geometry using the `base_geom_crs`):

In [ ]:
ba = pls.BufferAnalysis(
    input_filepath, base_geom, buffer_dists, base_geom_crs=base_geom_crs
)

The `BufferAnalysis` instance will automatically generate the three landscapes of interest (one for each buffer distance) by masking the pixels of the input raster:

In [ ]:
fig = ba.plot_landscapes(
    cmap=sls.noas04_4_cmap, show_kwargs=dict(norm=sls.noas04_4_norm)
)

Likewise `ZonalAnalysis`, we can compute the landscap metrics for each buffer distance with the `compute_class_metrics_df` and `compute_landscape_metrics_df` methods, e.g.:

In [ ]:
ba.compute_class_metrics_df()

Note that the data frames for `BufferAnalysis`, likewise those of `SpatioTemporalAnalysis` or `ZonalAnalysis` (see [above](#customizing-zonal-analysis)) can be customized via by passing the arguments `metrics`, `classes` or `metrics_kws` to the `compute_class_metrics_df` and `compute_landscape_metrics_df` methods.

The `plot_metric` method of `BufferAnalysis` will plot the value of a given metric for each of the buffered landscapes:

In [ ]:
ba.plot_metric("proportion_of_landscape", class_val=URBAN_CLASS_VAL)

The specific plot above shows how the proportion of landscape (y-axis) occupied by urban land uses diminishes with the buffer distance (x-axis) around the feature of interest (i.e., the city center of Chatel-St-Denis).

To examine more closely how landscape patterns change as we move along the urban-rural gradient, we might actually want to compute the metrics for each of the buffer rings that lie within each pair of increasing buffer distances. For instance, for the buffer distances considered in this example (i.e., 2000, 4000 and 6000), we would like to compute the metrics for the buffer rings that go from 0 to 2000m, 2000 to 4000m and 4000 to 6000m around the center of Chatel-St-Denis).

To that end, we might pass the argument `buffer_rings=True` when instantiating `BufferAnalysis` as in:

In [ ]:
ba = pls.BufferAnalysis(
    input_filepath,
    base_geom,
    buffer_dists,
    buffer_rings=True,
    base_geom_crs=base_geom_crs,
)
fig = ba.plot_landscapes(
    cmap=sls.noas04_4_cmap, show_kwargs=dict(norm=sls.noas04_4_norm)
)

In [ ]:
ba.compute_class_metrics_df()

In [ ]:
ba.plot_metric("proportion_of_landscape", class_val=URBAN_CLASS_VAL)

Again, the zonal statistics of a metric can be represented in space with of the `compute_zonal_statistics_gdf` method:

In [ ]:
metrics = ["proportion_of_landscape", "edge_density"]
zonal_statistics_gdf = ba.compute_zonal_statistics_gdf(
    metrics=metrics, class_val=URBAN_CLASS_VAL
)
for metric in metrics:
    ax = zonal_statistics_gdf.plot(metric, alpha=0.6, legend=True)
    ax.set_title(metric)
    cx.add_basemap(
        ax, crs=zonal_statistics_gdf.crs, source=cx.providers.CartoDB.Positron
    )

### From Polygon

We might as well build our buffer zones from polygon geometries such as administrative boundaries

In [ ]:
# the administrative boundaries have been geocoded with osmnx, i.e.,
# `ox.geocode_to_gdf("Chatel-St-Denis, Switzerland")`, and stored as a file so
# that building the docs does not require querying the nominatim API
gdf = gpd.read_file("data/chatel-st-denis.gpkg")
base_geom = gdf.geometry
base_geom.plot()

Note that in this case, since we are working with a GeoSeries that has a CRS defined, we do not need to set it explicitly with the `base_geom_crs` argument.

In [ ]:
base_geom.crs

Also note that since in this case our base geometry from which we will define buffer zones is already a polygon. Therefore, we might want to start from smaller buffer distances, even from zero, so that we start computing the metrics for the region defined by the polygon itself (in our example, the administrative boundaries)

In [ ]:
buffer_dists = [0, 1000, 2000]
ba = pls.BufferAnalysis(input_filepath, base_geom, buffer_dists)
fig = ba.plot_landscapes(
    cmap=sls.noas04_4_cmap, show_kwargs=dict(norm=sls.noas04_4_norm)
)

In [ ]:
ba.plot_metric("proportion_of_landscape", class_val=URBAN_CLASS_VAL)

## Zonal Grid Analysis

Another recurrent approach to zonal analysis is to decompose the landscape raster into a coarser grid and compute the landscape metrics for each zone cell. This is the purpose of the `ZonalGridAnalysis` class (which also inherits from `ZonalAnalysis`). We can instantiate it by providing size (in units of the landscape CRS) that we desire in each zone cell as in:

In [ ]:
zone_width, zone_height = 2000, 2000  # in this case, in meters

zga = pls.ZonalGridAnalysis(
    input_filepath,
    zone_width=zone_width,
    zone_height=zone_height,
)

Alternatively, we can instead define the number of zones that we desire in each dimension by means of the `num_zone_rows` and `num_zone_cols` keyword arguments of the initialization method.

The `ZonalGridAnalysis` class will automatically discard all the zone cells that have no data in the original raster. We can visualize the zonal grid (in random grid cell colors) as in:

In [ ]:
zga.plot_landscapes()

The `compute_class_metrics_df` and `compute_landscape_metrics_df` class operate exactly like in the other classes:

In [ ]:
zga.compute_class_metrics_df(metrics=metrics, classes=[URBAN_CLASS_VAL])

Note that the data frames are now indexed by a list of tuples that correspond to the row, column location of each zone.

Like in the other zonal analysis classes, the zonal metrics van be represented in space by means of the `compute_zonal_statistics_gdf` method. For instance, we can view the spatial distribution of the edge density at the landscape level as in:

In [ ]:
zonal_statistics_gdf = zga.compute_zonal_statistics_gdf(
    metrics=metrics, class_val=URBAN_CLASS_VAL
)
ax = zonal_statistics_gdf.plot("edge_density", alpha=0.6, legend=True)
cx.add_basemap(ax, crs=zonal_statistics_gdf.crs, source=cx.providers.CartoDB.Positron)

## See also

* [SpatioTemporalZonalAnalysis](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/04-spatiotemporal-zonal-analysis.ipynb)